# Diseño del Pipeline de Datos y Prompting
En este notebook se diseña el pipeline que conecta los datos preprocesados 
con el modelo generativo LLaMA 3.3 70B vía Groq para generar crónicas periodísticas automáticas.

In [13]:
import pandas as pd
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

df = pd.read_csv("../data/processed/epl_clean.csv")
print(f"Dataset cargado: {df.shape[0]} partidos")

Dataset cargado: 9325 partidos


## Construcción del prompt
Se diseña una función que traduce las estadísticas de un partido en un prompt 
estructurado para guiar al modelo en la generación de la crónica.

In [14]:
def build_prompt(row):
    resultado_map = {'H': f"ganó {row['HomeTeam']}", 'A': f"ganó {row['AwayTeam']}", 'D': "empate"}
    resultado_ht_map = {'H': f"ganaba {row['HomeTeam']}", 'A': f"ganaba {row['AwayTeam']}", 'D': "iban empatados"}

    favorito = row['Favorite']
    sorpresa = "Sí, fue una sorpresa" if row['Upset'] else "No, ganó el favorito o empató"

    prompt = f"""
Eres un periodista deportivo experto en fútbol inglés. 
Redactá una crónica periodística en español de aproximadamente 200 palabras 
sobre el siguiente partido de la Premier League.

La crónica debe tener un título atractivo, narrar el desarrollo del partido 
y destacar los datos más relevantes. Usá un tono dinámico y profesional.

--- DATOS DEL PARTIDO ---
Fecha: {row['MatchDate']}
Local: {row['HomeTeam']}
Visitante: {row['AwayTeam']}
Resultado final: {int(row['FTHome'])} - {int(row['FTAway'])} ({resultado_map[row['FTResult']]})
Al descanso: {int(row['HTHome'])} - {int(row['HTAway'])} ({resultado_ht_map[row['HTResult']]})

Estadísticas:
- Tiros (local / visitante): {int(row['HomeShots'])} / {int(row['AwayShots'])}
- Tiros al arco (local / visitante): {int(row['HomeTarget'])} / {int(row['AwayTarget'])}
- Corners (local / visitante): {int(row['HomeCorners'])} / {int(row['AwayCorners'])}
- Faltas (local / visitante): {int(row['HomeFouls'])} / {int(row['AwayFouls'])}
- Tarjetas amarillas (local / visitante): {int(row['HomeYellow'])} / {int(row['AwayYellow'])}
- Tarjetas rojas (local / visitante): {int(row['HomeRed'])} / {int(row['AwayRed'])}

Contexto:
- Favorito según cuotas: {favorito}
- ¿Fue sorpresa el resultado?: {sorpresa}
- Forma reciente local (últimos 5 partidos, puntos): {row['Form5Home']}
- Forma reciente visitante (últimos 5 partidos, puntos): {row['Form5Away']}
- Diferencia de Elo (local - visitante): {row['EloDiff']} puntos
"""
    return prompt

## Prueba con un partido
Generamos la crónica de un partido de prueba para validar el pipeline.

In [15]:
partido = df.iloc[100]
prompt = build_prompt(partido)

print("=== PROMPT ENVIADO ===")
print(prompt)

=== PROMPT ENVIADO ===

Eres un periodista deportivo experto en fútbol inglés. 
Redactá una crónica periodística en español de aproximadamente 200 palabras 
sobre el siguiente partido de la Premier League.

La crónica debe tener un título atractivo, narrar el desarrollo del partido 
y destacar los datos más relevantes. Usá un tono dinámico y profesional.

--- DATOS DEL PARTIDO ---
Fecha: 2000-10-28
Local: Aston Villa
Visitante: Charlton
Resultado final: 2 - 1 (ganó Aston Villa)
Al descanso: 2 - 0 (ganaba Aston Villa)

Estadísticas:
- Tiros (local / visitante): 12 / 7
- Tiros al arco (local / visitante): 7 / 4
- Corners (local / visitante): 9 / 6
- Faltas (local / visitante): 13 / 11
- Tarjetas amarillas (local / visitante): 3 / 2
- Tarjetas rojas (local / visitante): 0 / 0

Contexto:
- Favorito según cuotas: Aston Villa
- ¿Fue sorpresa el resultado?: No, ganó el favorito o empató
- Forma reciente local (últimos 5 partidos, puntos): 8.0
- Forma reciente visitante (últimos 5 partidos, pu

In [16]:
def generate_cronica(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500
    )
    return response.choices[0].message.content

partido = df.iloc[100]
prompt = build_prompt(partido)
cronica = generate_cronica(prompt)

print("=== CRÓNICA GENERADA ===")
print(cronica)

=== CRÓNICA GENERADA ===
**Aston Villa se impone con autoridad sobre Charlton**

En un partido emocionante en la Premier League, Aston Villa logró una victoria merecida sobre Charlton con un resultado final de 2-1. Desde el inicio, los locales demostraron su superioridad, llevándose la iniciativa y controlando el ritmo del juego. Al descanso, ya se encontraban 2-0 arriba, gracias a su destacada eficacia en ataque.

Las estadísticas del partido reflejan la dominancia de Aston Villa, con 12 tiros totales y 7 al arco, en comparación con los 7 tiros y 4 al arco de Charlton. Además, los locales tuvieron 9 corners, superando los 6 del equipo visitante. A pesar de que Charlton intentó reaccionar en la segunda mitad, solo pudo anotar un gol, lo que no fue suficiente para cambiar el curso del encuentro.

Con esta victoria, Aston Villa consolida su posición en la tabla, mientras que Charlton deberá seguir trabajando para mejorar su rendimiento. La diferencia de Elo de 114.4 puntos entre ambos eq

## Experimentación con estilos de prompt
Se comparan dos estilos narrativos para el mismo partido: uno técnico y uno dramático.
Esto permite evaluar cómo el diseño del prompt influye en la calidad y tono del contenido generado.

In [17]:
def build_prompt_tecnico(row):
    resultado_map = {'H': f"ganó {row['HomeTeam']}", 'A': f"ganó {row['AwayTeam']}", 'D': "empate"}
    resultado_ht_map = {'H': f"ganaba {row['HomeTeam']}", 'A': f"ganaba {row['AwayTeam']}", 'D': "iban empatados"}
    sorpresa = "Sí, fue una sorpresa" if row['Upset'] else "No, ganó el favorito o empató"

    return f"""
Eres un analista deportivo especializado en estadísticas de fútbol.
Redactá un informe técnico en español de aproximadamente 200 palabras sobre el siguiente partido.
El informe debe centrarse en los números: dominio territorial, eficacia ofensiva, disciplina táctica y contexto de forma.
Usá un tono objetivo y analítico, sin dramatismo.

--- DATOS DEL PARTIDO ---
Fecha: {row['MatchDate']}
Local: {row['HomeTeam']} | Visitante: {row['AwayTeam']}
Resultado final: {int(row['FTHome'])} - {int(row['FTAway'])} ({resultado_map[row['FTResult']]})
Al descanso: {int(row['HTHome'])} - {int(row['HTAway'])} ({resultado_ht_map[row['HTResult']]})
Tiros (local/visitante): {int(row['HomeShots'])} / {int(row['AwayShots'])}
Tiros al arco: {int(row['HomeTarget'])} / {int(row['AwayTarget'])}
Corners: {int(row['HomeCorners'])} / {int(row['AwayCorners'])}
Faltas: {int(row['HomeFouls'])} / {int(row['AwayFouls'])}
Amarillas: {int(row['HomeYellow'])} / {int(row['AwayYellow'])}
Rojas: {int(row['HomeRed'])} / {int(row['AwayRed'])}
Favorito: {row['Favorite']} | ¿Sorpresa?: {sorpresa}
Forma local (últ. 5): {row['Form5Home']} pts | Forma visitante: {row['Form5Away']} pts
Diferencia Elo: {row['EloDiff']} pts
"""

def build_prompt_dramatico(row):
    resultado_map = {'H': f"ganó {row['HomeTeam']}", 'A': f"ganó {row['AwayTeam']}", 'D': "empate"}
    resultado_ht_map = {'H': f"ganaba {row['HomeTeam']}", 'A': f"ganaba {row['AwayTeam']}", 'D': "iban empatados"}
    sorpresa = "Sí, fue una sorpresa" if row['Upset'] else "No, ganó el favorito o empató"

    return f"""
Eres un comentarista deportivo apasionado. 
Redactá una crónica dramática y emocionante en español de aproximadamente 200 palabras.
Usá metáforas, suspenso y lenguaje vívido. Destacá si hubo remontada, sorpresa o tarjetas rojas.

--- DATOS DEL PARTIDO ---
Fecha: {row['MatchDate']}
Local: {row['HomeTeam']} | Visitante: {row['AwayTeam']}
Resultado final: {int(row['FTHome'])} - {int(row['FTAway'])} ({resultado_map[row['FTResult']]})
Al descanso: {int(row['HTHome'])} - {int(row['HTAway'])} ({resultado_ht_map[row['HTResult']]})
Tiros (local/visitante): {int(row['HomeShots'])} / {int(row['AwayShots'])}
Tiros al arco: {int(row['HomeTarget'])} / {int(row['AwayTarget'])}
Corners: {int(row['HomeCorners'])} / {int(row['AwayCorners'])}
Faltas: {int(row['HomeFouls'])} / {int(row['AwayFouls'])}
Amarillas: {int(row['HomeYellow'])} / {int(row['AwayYellow'])}
Rojas: {int(row['HomeRed'])} / {int(row['AwayRed'])}
Favorito: {row['Favorite']} | ¿Sorpresa?: {sorpresa}
Forma local (últ. 5): {row['Form5Home']} pts | Forma visitante: {row['Form5Away']} pts
Diferencia Elo: {row['EloDiff']} pts
"""

partido = df.iloc[100]

print("=== ESTILO TÉCNICO ===")
cronica_tecnica = generate_cronica(build_prompt_tecnico(partido))
print(cronica_tecnica)

print("\n=== ESTILO DRAMÁTICO ===")
cronica_dramatica = generate_cronica(build_prompt_dramatico(partido))
print(cronica_dramatica)

=== ESTILO TÉCNICO ===
**Informe Técnico: Aston Villa vs. Charlton (28-10-2000)**

El partido disputado entre Aston Villa y Charlton el 28 de octubre de 2000 resultó en una victoria para el equipo local por 2-1. Analizando los números, observamos que Aston Villa dominó territorialmente, con 12 tiros frente a 7 del equipo visitante. La eficacia ofensiva también estuvo a favor del equipo local, con 7 tiros al arco contra 4 de Charlton.

En cuanto a la disciplina táctica, ambos equipos mostraron un nivel similar de intensidad, con 13 faltas cometidas por Aston Villa y 11 por Charlton. Sin embargo, el equipo local recibió 3 amarillas, mientras que el visitante recibió 2. La diferencia en corners también fue notable, con 9 corners para Aston Villa y 6 para Charlton.

En el contexto de forma, Aston Villa venía de una racha de 8 puntos en los últimos 5 partidos, mientras que Charlton acumulaba 10 puntos. La diferencia en el ranking Elo era significativa, con 114.4 puntos a favor del equipo lo

## Selección de partidos interesantes
Se filtran partidos con características narrativas destacadas: sorpresas, remontadas y alta intensidad (tarjetas rojas).

In [19]:
# Sorpresas: ganó el no favorito
sorpresas = df[df['Upset'] == True].sample(5, random_state=42)

# Remontadas: perdía al descanso pero ganó
remontadas = df[
    ((df['HTResult'] == 'A') & (df['FTResult'] == 'H')) |
    ((df['HTResult'] == 'H') & (df['FTResult'] == 'A'))
].sample(5, random_state=42)

# Alta intensidad: al menos una tarjeta roja
intensos = df[(df['HomeRed'] + df['AwayRed']) >= 1].sample(5, random_state=42)

print(f"Sorpresas encontradas: {len(df[df['Upset'] == True])}")
print(f"Remontadas encontradas: {len(df[((df['HTResult'] == 'A') & (df['FTResult'] == 'H')) | ((df['HTResult'] == 'H') & (df['FTResult'] == 'A'))])}")
print(f"Partidos con tarjeta roja: {len(df[(df['HomeRed'] + df['AwayRed']) >= 1])}")

Sorpresas encontradas: 1947
Remontadas encontradas: 416
Partidos con tarjeta roja: 1231


## Generación en batch
Se generan crónicas para 20 partidos seleccionados (sorpresas, remontadas e intensos) y se guardan en `reports/`.

In [ ]:
import time

seleccion = pd.concat([sorpresas, remontadas, intensos]).drop_duplicates().head(20)
os.makedirs("../reports", exist_ok=True)

resultados = []

for i, (idx, row) in enumerate(seleccion.iterrows()):
    prompt = build_prompt(row)
    cronica = generate_cronica(prompt)
    
    nombre_archivo = f"../reports/{row['MatchDate']}_{row['HomeTeam']}_vs_{row['AwayTeam']}.txt".replace(" ", "_")
    with open(nombre_archivo, "w", encoding="utf-8") as f:
        f.write(cronica)
    
    resultados.append({
        'MatchDate': row['MatchDate'],
        'HomeTeam': row['HomeTeam'],
        'AwayTeam': row['AwayTeam'],
        'FTResult': row['FTResult'],
        'Upset': row['Upset'],
        'archivo': nombre_archivo
    })
    
    print(f"[{i+1}/20] {row['HomeTeam']} vs {row['AwayTeam']} ({row['MatchDate']}) ✓")
    time.sleep(3)  # evitar rate limit

df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv("../reports/indice_cronicas.csv", index=False)
print("\nBatch completado. Índice guardado en reports/indice_cronicas.csv")

[1/20] Fulham vs Leeds (2021-03-19) ✓
[2/20] Leicester vs Man City (2018-12-26) ✓
[3/20] QPR vs Stoke (2013-04-20) ✓
[4/20] Swansea vs Southampton (2014-05-03) ✓
[5/20] Blackburn vs Man United (2004-05-01) ✓
[6/20] West Brom vs Birmingham (2010-09-18) ✓
[7/20] West Ham vs Arsenal (2019-12-09) ✓
[8/20] Fulham vs Chelsea (2025-04-20) ✓
[9/20] Charlton vs Aston Villa (2006-12-30) ✓
[10/20] West Ham vs Blackburn (2005-08-13) ✓
[11/20] West Brom vs Man City (2010-11-07) ✓
[12/20] Burnley vs West Brom (2021-02-20) ✓
[13/20] Bolton vs Fulham (2005-04-09) ✓
[14/20] Bolton vs Newcastle (2010-11-20) ✓
[15/20] Liverpool vs Man United (2023-12-17) ✓

Batch completado. Índice guardado en reports/indice_cronicas.csv
